In [ ]:
# -*- coding: utf-8 -*-
# =========[ 0) Mount & Imports ]=========
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, random, pickle, zipfile
from typing import Dict, List
import numpy as np
from PIL import Image
from contextlib import contextmanager

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

logsoft = nn.LogSoftmax(dim=1)
NUM_WORKERS = 0

# =========[ 1) paths]=========
BASE = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark"
ZIP_PATH = f"{BASE}/tiny-imagenet-processed.zip"
DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

MODEL_PATH = f"{BASE}/Gtask7_best_test_for_finetune_ResNet18_GN_temp2.pth"

SAVE_FISHER_FULL    = f"{BASE}/expriment2_GN_task7F_tiny_imageNet_temp2.pkl"
SAVE_TOPK_PATH      = f"{BASE}/expriment2_GN_task7F_tiny_imageNet_topk_temp2.pkl"
SAVE_NEIGHBORS_PATH = f"{BASE}/expriment2_GN_task7F_tiny_imageNet_neighbors_temp2.pkl"

TOP_K = 500000
NUM_FISHER_SAMPLES = 2000
BATCH_FISHER = 32

os.makedirs(BASE, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")
    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(
           f"ZIP not found at:\n{zip_path}\nPlease update ZIP_PATH to the correct ZIP file path."
        )
    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"
    print(f"[INFO] Extracted. processed/ ready at: {processed_dir}")
    return data_root

DATA_ROOT = ensure_extracted(ZIP_PATH, DATA_ROOT)

# =========[ 3) Dataset (processed .npy) + Transforms ]=========
TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):

    def __init__(self, root: str, train: bool=True, transform: transforms = None):
        self.root = root
        self.train = train
        self.transform = transform
        split = "train" if self.train else "val"
        xs, ys = [], []
        for num in range(20):
            xs.append(np.load(os.path.join(root, f'processed/x_{split}_{num+1:02d}.npy')))
            ys.append(np.load(os.path.join(root, f'processed/y_{split}_{num+1:02d}.npy')))
        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])
        img = Image.fromarray(np.uint8(255 * img))
        if self.transform is not None:
            img = self.transform(img)
        return img, target

tf_fisher = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
])

def get_tiny_datasets_for_fisher():
    train_set = TinyImagenet(root=DATA_ROOT, train=True,  transform=tf_fisher)
    val_set   = TinyImagenet(root=DATA_ROOT, train=False, transform=tf_fisher)
    print(f"[INFO] Train size: {len(train_set)} | Val size: {len(val_set)}")
    return train_set, val_set

def build_tasks(num_classes: int = 200, classes_per_task: int = 20) -> Dict[str, List[int]]:
    assert num_classes % classes_per_task == 0
    n_tasks = num_classes // classes_per_task  # 10
    tasks = {}
    for t in range(n_tasks):
        start = t * classes_per_task
        tasks[f"task{t+1}"] = list(range(start, start + classes_per_task))
    return tasks

def filter_indices_by_classes(dataset, classes: List[int]) -> List[int]:
    t = dataset.targets
    return [i for i, y in enumerate(t) if int(y) in classes]

def remap_labels(original_targets: List[int], keep_classes: List[int]) -> List[int]:
    class_to_new = {c: i for i, c in enumerate(sorted(keep_classes))}
    return [class_to_new[int(y)] for y in original_targets]

class RelabeledSubset(Subset):
    def __init__(self, dataset, indices: List[int], keep_classes: List[int]):
        super().__init__(dataset, indices)
        original_targets = [int(dataset.targets[i]) for i in indices]
        self.new_targets = remap_labels(original_targets, keep_classes)

    def __getitem__(self, idx):
        x, _ = super().__getitem__(idx)
        y_new = self.new_targets[idx]
        return x, y_new
    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]

def make_task_subset(dataset, task_classes: List[int]) -> RelabeledSubset:
    idx = filter_indices_by_classes(dataset, task_classes)
    return RelabeledSubset(dataset, idx, task_classes)

def get_random_subset(dataset, n_samples=2000, seed=42):
    rng = random.Random(seed)
    n = len(dataset)
    k = min(n_samples, n)
    idxs = rng.sample(range(n), k)
    return Subset(dataset, idxs)

# =========[ 5) ResNet18 (GroupNorm) + Multi-Head 10×20 ]=========
def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

def _gn(num_channels: int, num_groups: int = 32):
    return nn.GroupNorm(num_groups, num_channels)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1   = _gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2   = _gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                _gn(planes * self.expansion)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()
        self.nf = nf
        self.in_planes = nf
        self.expansion = 1
        self.conv1 = conv3x3(3, nf)
        self.gn1   = _gn(nf)
        self.layer1 = self._make_layer(BasicBlock, nf,     2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, nf * 2, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, nf * 4, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, nf * 8, 2, stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers, in_planes = [], self.in_planes
        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion
        self.in_planes = in_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out)
        out = self.layer3(out); out = self.layer4(out)
        out = torch.nn.functional.avg_pool2d(out, out.shape[2])
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self) -> int:
        return self.nf * 8 * self.expansion  # 512

# ======== [MultiHeadNet] ========
class MultiHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()

    def add_head(self, task_name: str, num_classes: int):
        if task_name in self.heads:
            raise ValueError(f"Head '{task_name}' already exist.")
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[task_name] = head

    def forward(self, x: torch.Tensor, task_name: str = "task1") -> torch.Tensor:
        if task_name not in self.heads:
            raise ValueError(f"Head '{task_name}'  not found.")
        feat = self.backbone(x)
        return self.heads[task_name](feat)

# =========[ 6) Tools: ravel/unravel (neighbors) ]=========
def unravel_index(index, shape):
    dims = []
    for s in reversed(shape):
        dims.append(index % s)
        index //= s
    return tuple(reversed(dims))

def ravel_index(multi_idx, shape):
    flat = 0
    for idx, dim in zip(multi_idx, shape):
        flat = flat * dim + idx
    return flat

@contextmanager
def bn_eval_only(model: nn.Module):

    touched = []
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            touched.append((m, m.training))
            m.eval()
    try:
        yield
    finally:
        for m, was_training in touched:
            m.train(was_training)

# =========[ 8) Fisher (excluding heads.*) — with general automatic remapping ]=========
def compute_fisher_per_sample(model, dataloader, device, head_name="task7", exclude_heads=True, auto_remap=True):
    """
    Computes Fisher for all parameters except the heads (heads.*) if exclude_heads=True.
    - Only the specified head is used in the forward pass to compute the loss.
    - BatchNorm running statistics are frozen during computation (bn_eval_only).
    - General automatic remapping: if labels are in their original range (20*t..20*t+19), they are remapped to 0..19.
      (If the data has already been remapped to 0..19, nothing changes.)
    """
    was_training = model.training
    model.train()

    print(f"[DEBUG] computing Fisher for head={head_name}")

    if head_name not in model.heads:
        raise ValueError(f"Head '{head_name}' not found.")
    num_classes = int(model.heads[head_name].out_features)

    task_offset = 0
    if auto_remap and head_name.startswith("task"):
        try:
            tnum = int(head_name.replace("task", ""))
            task_offset = (tnum - 1) * 20
        except:
            task_offset = 0

    param_map = dict(model.named_parameters())

    backup_req = {}
    if exclude_heads:
        for n, p in param_map.items():
            if n.startswith("heads.") and p.requires_grad:
                backup_req[n] = True
                p.requires_grad_(False)

    def include_param(name, p):
        if exclude_heads and name.startswith("heads."):
            return False
        return p.requires_grad

    fisher = {
        n: torch.zeros_like(p, device=device)
        for n, p in param_map.items()
        if include_param(n, p)
    }

    total_samples = 0

    with bn_eval_only(model):
        print("[BN] Running stats frozen during Fisher computation.")
        for inputs, targets in dataloader:
            inputs  = inputs.to(device)
            targets = targets.to(device, dtype=torch.long)

            if total_samples == 0:
                print(f"[DEBUG] First batch BEFORE remap: {int(targets.min())} → {int(targets.max())}")

            if auto_remap and (targets.min() >= task_offset) and (targets.max() <= task_offset + (num_classes - 1)):
                targets = targets - task_offset

            if total_samples == 0:
                print(f"[DEBUG] First batch AFTER  remap: {int(targets.min())} → {int(targets.max())}")

            tmin = int(targets.min().item()); tmax = int(targets.max().item())
            if tmin < 0 or tmax >= num_classes:
                raise RuntimeError(
                    f"[Fisher] Found labels in batch outside 0..{num_classes-1} for head '{head_name}' "
                    f"(min={tmin}, max={tmax}). check subset/ReLabel."
                )

            for ex, lab in zip(inputs, targets):
                ex  = ex.unsqueeze(0)
                lab = lab.unsqueeze(0)

                model.zero_grad(set_to_none=True)

                logits = model(ex, head_name)

                yv = int(lab.item())
                if yv < 0 or yv >= num_classes:
                    raise RuntimeError(
                        f"[Fisher] Label {yv} out of range for head '{head_name}' "
                        f"(expected 0..{num_classes-1})."
                    )

                loss_i = -F.nll_loss(logsoft(logits), lab, reduction='none')  # log p(y|x)
                p_i = torch.exp(loss_i.detach())

                loss_i.mean().backward()

                w = float(p_i.mean().item())
                for name, p in param_map.items():
                    if name in fisher and p.grad is not None:
                        fisher[name] += w * (p.grad.detach() ** 2)

                total_samples += 1

    for name in fisher:
        fisher[name] /= max(1, total_samples)

    if backup_req:
        for n in backup_req.keys():
            param_map[n].requires_grad_(True)

    if not was_training:
        model.eval()

    return fisher

# =========[ 9) Top-K & neighbors ]=========
def get_topk_fisher_weights(fisher_dict, model_state_dict, k):
    if k <= 0:
        return []
    all_entries = []
    for name, f in fisher_dict.items():
        flat_f = f.flatten()
        flat_w = model_state_dict[name].flatten()
        for i in range(flat_f.numel()):
            all_entries.append({
                "name": name,
                "index": i,
                "value": float(flat_w[i].item()),
                "fisher": float(flat_f[i].item())
            })
    all_entries.sort(key=lambda x: x["fisher"], reverse=True)
    return all_entries[:k]

def extract_conv_neighbors(topk_entries, model, fisher_dict):
    neighbors = []
    if not topk_entries:
        return neighbors
    param_shapes = {name: p.shape for name, p in model.named_parameters()}
    fisher_flat = {name: tens.flatten() for name, tens in fisher_dict.items()}

    neighbor_offsets = [
        (0, 0, -1, -1), (0, 0, -1,  1),
        (0, 0,  1, -1), (0, 0,  1,  1),
        (0, 0, -1,  0), (0, 0,  1,  0),
        (0, 0,  0, -1), (0, 0,  0,  1),
    ]
    topk_set = set((e["name"], e["index"]) for e in topk_entries)
    added = set()

    for e in topk_entries:
        name, flat_idx = e["name"], e["index"]
        shape = param_shapes[name]
        if len(shape) != 4:  # only Conv layers
            continue
        oc, ic, kh, kw = unravel_index(flat_idx, shape)
        for do, di, dh, dw in neighbor_offsets:
            no, ni, nh, nw = oc + do, ic + di, kh + dh, kw + dw
            if 0 <= no < shape[0] and 0 <= ni < shape[1] and 0 <= nh < shape[2] and 0 <= nw < shape[3]:
                n_flat = ravel_index((no, ni, nh, nw), shape)
                key = (name, n_flat)
                if key in topk_set or key in added:
                    continue
                neighbors.append({
                    "name": name,
                    "index": n_flat,
                    "position": (no, ni, nh, nw),
                    "fisher": float(fisher_flat[name][n_flat].item())
                })
                added.add(key)
    return neighbors

# =========[ 10) Build model, load checkpoint (GN) ]=========
backbone = ResNet18Backbone(nf=64).to(DEVICE)
model = MultiHeadNet(backbone=backbone).to(DEVICE)
for i in range(1, 11):
    model.add_head(f"task{i}", num_classes=20)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print("[INFO] Missing keys (not loaded):", missing)
if unexpected:
    print("[INFO] Unexpected keys (ignored):", unexpected)
model.train()
print("Loaded checkpoint:", MODEL_PATH)

# =========[ 11) Prepare Fisher samples from task7 only ]=========
tasks = build_tasks(num_classes=200, classes_per_task=20)
task7_classes = tasks["task7"]  # [120..139]

train_set, _ = get_tiny_datasets_for_fisher()

#  This automatically remaps labels to 0..19
task7_train = make_task_subset(train_set, task7_classes)

# Check 1: original classes inside the subset before remapping
labels = [int(y) for y in np.unique([train_set.targets[i] for i in task7_train.indices])]
print("✅ Original classes inside task7_train:", labels)

subset_200 = get_random_subset(task7_train, n_samples=NUM_FISHER_SAMPLES, seed=SEED)
subset_loader = DataLoader(subset_200, batch_size=BATCH_FISHER,
                           shuffle=False, num_workers=NUM_WORKERS)

# ✅ Check 2: label range inside the DataLoader (after remapping)
for x, y in subset_loader:
    print("✅ Label range inside subset_200 (after remapping):", y.min().item(), "→", y.max().item())
    break

# =========[ 12) Fisher on backbone only (excluding heads) + save + TopK/Neighbors + statistics ]=========
# --- Use task7 head + general fallback remapping inside the function in case labels 120..139 are passed by mistake ---
fisher_info = compute_fisher_per_sample(
    model,
    subset_loader,
    DEVICE,
    head_name="task7",
    exclude_heads=True,
    auto_remap=True
)

# Verify that no task-head parameters are included in Fisher
heads_in_fisher = [k for k in fisher_info.keys() if k.startswith("heads.")]
print("[CHECK] heads in fisher:", len(heads_in_fisher))

with open(SAVE_FISHER_FULL, "wb") as f:
    pickle.dump({k: v.cpu() for k, v in fisher_info.items()}, f)
print("Saved full Fisher to:", SAVE_FISHER_FULL)

topk_info = get_topk_fisher_weights(fisher_info, model.state_dict(), TOP_K)
neighbors_info = extract_conv_neighbors(topk_info, model, fisher_info)

with open(SAVE_TOPK_PATH, "wb") as f:
    pickle.dump(topk_info, f)
with open(SAVE_NEIGHBORS_PATH, "wb") as f:
    pickle.dump(neighbors_info, f)

print(f"✅ Fisher computed from {len(subset_200)} samples (per-sample, diagonal) on Tiny-ImageNet task7 (20 classes) excluding heads.")
print(f"✅ Top-K total: {len(topk_info)}")
conv_topk = [e for e in topk_info if len(model.state_dict()[e['name']].shape) == 4] if topk_info else []
if topk_info:
    print(f"📌 Top-K from Conv2D: {len(conv_topk)} → {100*len(conv_topk)/len(topk_info):.2f}%")
else:
    print("📌 Top-K from Conv2D: 0")
print(f"📌 Neighbors extracted from Conv2D: {len(neighbors_info)}")

print("\n📊 Fisher Statistics per-parameter:")
print(f"{'Parameter':40s} | {'Mean':>12s} | {'Min':>12s} | {'Max':>12s}")
print("-"*85)
for name, tens in fisher_info.items():
    vals = tens.detach().cpu().view(-1)
    mean_val = vals.mean().item()
    min_val  = vals.min().item()
    max_val  = vals.max().item()
    print(f"{name:40s} | {mean_val:12.4e} | {min_val:12.4e} | {max_val:12.4e}")

print("✔️ Done.")



Mounted at /content/drive
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
Loaded checkpoint: /content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gtask7_best_test_for_finetune_ResNet18_GN_temp2.pth
[INFO] Train size: 100000 | Val size: 10000
✅ الفئات الأصلية داخل task7_train: [120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139]
✅ نطاق الليبلز داخل subset_200 (بعد الريماب): 0 → 19
[DEBUG] computing Fisher for head=task7
[BN] Running stats frozen during Fisher computation.
[DEBUG] First batch BEFORE remap: 0 → 19
[DEBUG] First batch AFTER  remap: 0 → 19
[CHECK] heads in fisher: 0
Saved full Fisher to: /content/drive/MyDrive/ML_Project/project_files/First_benchmark/expriment2_GN_task7F_tiny_imageNet_temp2.pkl
✅ Fisher computed from 2000 samples (per-sample, diagonal) on Tiny-ImageNet task7 (20 classes) excluding heads.
✅ Top-K total: 500000
📌 Top-K from Conv2D: 490998 → 98.20%
📌 Neighbors e